Question 1a: find out number of requests per category 

In [1]:
import pandas as pd
import geopandas as gpd

#read data (.csv velech chli irrefüehrend wöu s isch kes csv meh sondern es pandas (= datestruktur))
meldungen_csv = gpd.read_file("data/raw/stzh.zwn_meldungen_p.json")

#check
#display(meldungen_csv)

#eine leere Spalte an meldungen_csv anfügen 
meldungen_csv["anzahl_meldungen"] = 0
#display(meldungen_csv)

#leere Spalte mit Werten füllen
for kategorie in meldungen_csv["service_code"].unique():  #dür aui kategorie düregoo
    anzahl_meldungen = meldungen_csv[meldungen_csv["service_code"] == kategorie].shape[0] #azahl zile i dere kategorie zöue

    meldungen_csv.loc[ #mäudige i neui Spalte schribe,, loc wöu select integer based on position (glaub????)
        meldungen_csv["service_code"] == kategorie, 
        "anzahl_meldungen"] = anzahl_meldungen
    
    print(f"Kategorie: {kategorie}, Anzahl Meldungen: {anzahl_meldungen}")
    
#display(meldungen_csv)



Kategorie: Strasse/Trottoir/Platz, Anzahl Meldungen: 9870
Kategorie: Abfall/Sammelstelle, Anzahl Meldungen: 27339
Kategorie: Grünflächen/Spielplätze, Anzahl Meldungen: 7238
Kategorie: Beleuchtung/Uhren, Anzahl Meldungen: 5407
Kategorie: Graffiti, Anzahl Meldungen: 3759
Kategorie: Signalisation/Lichtsignal, Anzahl Meldungen: 10975
Kategorie: Brunnen/Hydranten, Anzahl Meldungen: 1289
Kategorie: VBZ/ÖV, Anzahl Meldungen: 1882
Kategorie: Allgemein, Anzahl Meldungen: 3969
Kategorie: Schädlinge, Anzahl Meldungen: 895


Question 1b: find out number of requests per category and Kreis/Quartier 
- improvements: listen sortieren nach anzahl meldungen

In [2]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd

#read data 
quartiere_json = gpd.read_file("data/raw/stzh.adm_statistische_quartiere_v.json")
quartiere_csv = gpd.read_file("data/raw/stzh.adm_statistische_quartiere_v.csv")
quartiere_ch = quartiere_json.to_crs(epsg = 2056)
print("Quartiere JSON:", quartiere_json.columns)
print("Quartiere CSV:", quartiere_ch.columns)

#merge quartiere json und csv um sowohl räumliche daten als auch attribute wie kname zu haben
quartiere_gdf = quartiere_json.merge(
    quartiere_csv, on = "objid", how = "left")
quartiere_gdf = quartiere_gdf.set_geometry("geometry_x") #aktive geometrie wieder definieren
quartiere_gdf = quartiere_gdf.to_crs(epsg = 2056)
print("Quartiere GDF:", quartiere_gdf.columns)

meldungen_gdf = gpd.GeoDataFrame(
    meldungen_csv,
    geometry=gpd.points_from_xy(meldungen_csv["e"], meldungen_csv["n"]),
    crs=quartiere_ch.crs)
meldungen_ch = meldungen_gdf.to_crs(epsg = 2056)

#crs prüfen
print("CRS Quartiere", quartiere_gdf.crs)
print("CRS Meldungen", meldungen_ch.crs)

#spatial join
meldungen_quartier_join = gpd.sjoin(
    meldungen_ch, #left
    quartiere_gdf, #right
    how = "inner", predicate = "intersects")
print("JOIN:", meldungen_quartier_join.columns)


#meldungen pro quartier und kreis 
meldungen_pro_quartier = (meldungen_quartier_join.groupby("qname").size().reset_index(name = "anzahl_meldungen_quartier"))
display(meldungen_pro_quartier)

meldungen_pro_kreis = (meldungen_quartier_join.groupby("kname").size().reset_index(name = "anzahl_meldungen_kreis"))
display(meldungen_pro_kreis)

#neue spalten zu join hinzufügen
join_updated = meldungen_quartier_join.merge(
    meldungen_pro_kreis,
    on = "kname",
    how = "left")

join_updated = join_updated.merge(
    meldungen_pro_quartier,
    on = "qname",
    how = "left")

join_updated = join_updated[["objectid", "requested_datetime", "e", "n", "service_code", "geometry", "anzahl_meldungen", "index_right", "objid", "objectid_x", "objectid_y", "qname", "qnr", "kname", "knr", "geometry_y", "anzahl_meldungen_kreis"]]
print(join_updated.head())
print("Join Updated:", join_updated.columns)

Quartiere JSON: Index(['objid', 'objectid', 'geometry'], dtype='str')
Quartiere CSV: Index(['objid', 'objectid', 'geometry'], dtype='str')
Quartiere GDF: Index(['objid', 'objectid_x', 'geometry_x', 'objectid_y', 'qname', 'qnr',
       'kname', 'knr', 'geometry_y'],
      dtype='str')
CRS Quartiere EPSG:2056
CRS Meldungen EPSG:2056
JOIN: Index(['objectid', 'service_request_id', 'requested_datetime',
       'agency_sent_datetime', 'updated_datetime', 'e', 'n', 'service_code',
       'service_name', 'status', 'userid', 'title', 'detail', 'media_url',
       'interface_used', 'service_notice', 'description', 'url', 'geometry',
       'anzahl_meldungen', 'index_right', 'objid', 'objectid_x', 'objectid_y',
       'qname', 'qnr', 'kname', 'knr', 'geometry_y'],
      dtype='str')


,qname,anzahl_meldungen_quartier
0,Affoltern,2419
1,Albisrieden,2048
2,Alt-Wiedikon,2502
3,Altstetten,4094
4,City,1741
5,Enge,2756
6,Escher Wyss,1562
7,Fluntern,1254
8,Friesenberg,1415
9,Gewerbeschule,2230


,kname,anzahl_meldungen_kreis
0,Kreis 1,5738
1,Kreis 10,6394
2,Kreis 11,8026
3,Kreis 12,2764
4,Kreis 2,6225
5,Kreis 3,9152
6,Kreis 4,10569
7,Kreis 5,3792
8,Kreis 6,5110
9,Kreis 7,5714


   objectid requested_datetime        e        n            service_code  \
0         1     20130314151615  2678968  1247548  Strasse/Trottoir/Platz   
1         2     20130314151757  2680746  1249916  Strasse/Trottoir/Platz   
2         3     20130315091416  2684605  1251431  Strasse/Trottoir/Platz   
3         4     20130315091715  2681754  1250376  Strasse/Trottoir/Platz   
4         5     20130315103653  2683094  1247762     Abfall/Sammelstelle   

                  geometry  anzahl_meldungen  index_right objid  objectid_x  \
0  POINT (2678968 1247548)              9870           16    23          16   
1  POINT (2680746 1249916)              9870           20    28          21   
2  POINT (2684605 1251431)              9870           33     8          33   
3  POINT (2681754 1250376)              9870           21    29          22   
4  POINT (2683094 1247762)             27339           15    25          18   

  objectid_y        qname  qnr     kname knr  \
0         16  Albisr

Question 2: requests in kreisen around the lake (difference between different lakesides)
"close to the lake" = Kreise 1,2,8 (chinawiese = kreis 8)

In [3]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

#read file
quartiere_gdf
quartiere_subset = quartiere_gdf[["objid", "geometry_x", "kname", "knr"]]


fläche_csv = pd.read_csv("data/raw/bevölkerung_zh2.csv")
print("Fläche:", fläche_csv.columns)
fläche_subset = fläche_csv[fläche_csv["RaumLang"].str.contains("Kreis", na = False)]
#display(fläche_subset)
#display(fläche_csv)

#Join Fläche und sonstige Geoinfos
join_fläche_meldungen = fläche_subset.merge(
    join_updated[["kname", "anzahl_meldungen_kreis"]],
    left_on = "RaumLang", 
    right_on = "kname", 
    how = "left")

#Meldungsdichte pro Hektar
join_fläche_meldungen["meldungsdichte_kreis"] = join_fläche_meldungen["anzahl_meldungen_kreis"]/join_fläche_meldungen["FlaecheT"]
display(join_fläche_meldungen) 


Fläche: Index(['RaumKategorie', 'RaumSort', 'RaumLang', 'StichtagDatJahr', 'FlaecheT',
       'FlaecheL', 'FlaecheS'],
      dtype='str')


,RaumKategorie,RaumSort,RaumLang,StichtagDatJahr,FlaecheT,FlaecheL,FlaecheS,kname,anzahl_meldungen_kreis,meldungsdichte_kreis
0,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
1,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
2,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
3,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
4,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
...,...,...,...,...,...,...,...,...,...,...
1234586,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920
1234587,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920
1234588,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920
1234589,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920


Question 2: visualization 
GEIT NED, STÜRZT AB SOBAUD IS WETT PLOTTE

In [ ]:
#Choropleth Map
fig, ax = plt.subplots(figsize=(10, 8)) #set up figure and axes

# Plot the choropleth map using the population column
quartiere_meldungsdichte = quartiere_subset.merge(
    join_fläche_meldungen[["kname", "meldungsdichte_kreis"]],
    on = "kname",
    how = "left")

#choropleth map 
fig, ax = plt.subplots(figsize=(10,8))

dmin = quartiere_meldungsdichte["meldungsdichte_kreis"].quantile(0.02)
dmax = quartiere_meldungsdichte["meldungsdichte_kreis"].quantile(0.98)

legend_options = {
    "label": "Report Density per Stadtkreis Normalized per Population",
    "orientation": "horizontal",
    "shrink": 0.6,
    "pad" : 0.05}

charte_plot = quartiere_meldungsdichte.plot(
    ax = ax, 
    column = "meldungsdichte_kreis",
    dmin = dmin,
    dmax = dmax,
    cmap = "viridis",
    legend = True,
    legend_kwds = legend_options,
    edgecolor = "grey",
    linewidth = 0.1
)



Question 3: 

In [ ]:
import geopandas as gpd
import pandas as pd


#meldungen quartier join
#print(join_updated.head())
join_updated["requested_datetime"] = pd.to_datetime(
    join_updated["requested_datetime"], format = "%Y%m%d%H%M%S")

#close to lake area
kreise_see = join_updated[join_updated["knr"].isin(["1", "2", "8"])]
display(kreise_see)

#check
meldung_jahr = join_updated["requested_datetime"].dt.year
maldung_datum = join_updated["requested_datetime"].dt.date



,objectid,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,...,objid,objectid_x,objectid_y,qname,qnr,kname,knr,geometry_y,anzahl_meldungen_kreis,anzahl_meldungen_quartier
4,5,6,2013-03-15 10:36:53,20130422182505,20130423135033,2683094,1247762,Abfall/Sammelstelle,Abfall/Sammelstelle,fixed - council,...,25,18,18,City,14,Kreis 1,1,"POLYGON ((2682373.2 1247057,2682399.5 1247083....",5738,1741
5,6,7,2013-03-16 17:54:42,20130328071005,20130412080944,2683475,1247422,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,21,14,14,Rathaus,11,Kreis 1,1,"POLYGON ((2683316.2 1247632.6,2683319.8 124763...",5738,1457
6,7,8,2013-03-16 18:04:21,20130506155005,20130506172911,2683303,1247675,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,31,25,25,Lindenhof,13,Kreis 1,1,"POLYGON ((2683037 1247571.6,2683044.5 1247600,...",5738,1125
14,15,16,2013-03-21 09:06:05,20130321093007,20130412082531,2684556,1245435,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,18,10,10,Mühlebach,82,Kreis 8,8,"POLYGON ((2683784.2 1246610.5,2683801.5 124663...",2997,908
19,20,22,2013-03-27 10:34:48,20130328071004,20130412084327,2683538,1247650,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,21,14,14,Rathaus,11,Kreis 1,1,"POLYGON ((2683316.2 1247632.6,2683319.8 124763...",5738,1457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72612,72613,81169,2026-05-01 07:48:09,20260503151706,20260503151706,2682625,1247058,Signalisation/Lichtsignal,Signalisation/Lichtsignal,confirmed,...,25,18,18,City,14,Kreis 1,1,"POLYGON ((2682373.2 1247057,2682399.5 1247083....",5738,1741
72613,72614,81170,2026-05-01 08:39:59,20260503150705,20260503150705,2683133,1247858,Brunnen/Hydranten,Brunnen/Hydranten,confirmed,...,31,25,25,Lindenhof,13,Kreis 1,1,"POLYGON ((2683037 1247571.6,2683044.5 1247600,...",5738,1125
72615,72616,81188,2026-05-01 16:23:20,20260503152205,20260503152205,2682876,1244477,Grünflächen/Spielplätze,Grünflächen/Spielplätze,confirmed,...,16,8,8,Wollishofen,21,Kreis 2,2,"POLYGON ((2681417 1244793.5,2681418 1244817.4,...",6225,2837
72619,72620,81201,2026-05-01 23:33:55,20260503150206,20260503150206,2682506,1244158,Allgemein,Allgemein,confirmed,...,16,8,8,Wollishofen,21,Kreis 2,2,"POLYGON ((2681417 1244793.5,2681418 1244817.4,...",6225,2837
